# AlphaTrain pillar3g — V14 distillation with target sharpening

Next iteration past **pillar3f** (the task-arithmetic best, HISTORY 167). Train on the V14 corpus generated by pillar3f + MCTS@400 (selfplay) and pillar3f + MCTS@600/400 (crisis). Apply the same target_temperature=0.5 sharpening recipe that gave +57% policy lift over baseline (HISTORY 153-156).

## Setup

| param | value | rationale |
|---|---|---|
| corpus | V14 (9.69M states) | selfplay 960 games + crisis 2,608 files |
| warm-start | pillar3f (pillar3f.pt) | task-arithmetic best (5k median 22,181) |
| epochs | 20 | pillar3b recipe (peaked ~ep20 — eval all epochs) |
| batch-size | 32,768 | H100/G4 friendly |
| lr | 3e-4, warmup 1 epoch | |
| target_temperature | **0.5** | V14 top1 mean (T=1.0) = 0.249, matches V12's 0.26 → same recipe as pillar3f |
| color augmentation | on (default) | 7! symmetry — +4% lift (HISTORY 143-144) |
| dihedral augmentation | on (default) | 8× board symmetry |

Decision gate at eval time: beat pillar3f's 5k bar (median 22,181 / mean 31,617), judged on median + floor, eval **uncapped** (`eval_policy`, seeds 775000-779999).

## Upload to Drive (`MyDrive/alphatrain/`)

1. `colorlines_pillar3d_v2.tar.gz` — code tarball (~1.2 MB)
2. `v14_pillar3f.pt.gz` — V14 corpus, gzipped (build on M5: `build_expert_v2_tensor --games-dir data/selfplay_v14 data/crisis_v14_s600 --policy-only-data --output alphatrain/data/v14_pillar3f.pt` then `gzip -k`)
3. `pillar3f.pt` — pillar3f warm-start (~45 MB)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil, time

DRIVE = '/content/drive/MyDrive/alphatrain'

!cp {DRIVE}/colorlines_pillar3d_v2.tar.gz /content/
!cd /content && tar xzf colorlines_pillar3d_v2.tar.gz

os.makedirs('/content/alphatrain/data', exist_ok=True)
shutil.copy(f'{DRIVE}/pillar3f.pt',
            '/content/alphatrain/data/pillar3f.pt')
print(f'Warm-start (pillar3f): {os.path.getsize("/content/alphatrain/data/pillar3f.pt")/1e6:.0f} MB')

# Copy gzipped tensor to local disk FIRST, then verify size, then decompress.
# Streaming through Drive FUSE (`gzip -dc {DRIVE}/... > local`) is unreliable
# for files this size: FUSE can serve partial data and truncate the .pt silently.
t0 = time.time()
!cp {DRIVE}/v14_pillar3f.pt.gz /content/v14_pillar3f.pt.gz
gz_size = os.path.getsize('/content/v14_pillar3f.pt.gz')
print(f'.gz copied: {gz_size:,} bytes ({gz_size/1e6:.0f} MB)')
EXPECTED_GZ = 671_908_042
assert EXPECTED_GZ is None or gz_size == EXPECTED_GZ, (
    f'.gz on Drive is truncated! got {gz_size} expected {EXPECTED_GZ}. '
    f'Re-upload v14_pillar3f.pt.gz to Drive before retrying.')
!gunzip -t /content/v14_pillar3f.pt.gz && echo '.gz integrity OK'
!gzip -dc /content/v14_pillar3f.pt.gz > /content/alphatrain/data/v14_pillar3f.pt
pt_size = os.path.getsize('/content/alphatrain/data/v14_pillar3f.pt')
EXPECTED_PT = 5_748_249_183
assert (EXPECTED_PT is None and pt_size > 1_000_000_000) or pt_size == EXPECTED_PT, (
    f'Decompressed .pt size wrong! got {pt_size} expected {EXPECTED_PT}.')
print(f'V14 tensor: {pt_size/1e9:.2f} GB ({time.time()-t0:.0f}s)')
# Optional: free disk by removing the .gz now that the .pt is verified.
!rm /content/v14_pillar3f.pt.gz

!pip install -q numpy numba scipy

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    g = torch.cuda.get_device_properties(0)
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {g.total_memory / 1e9:.1f} GB')

## Train pillar3g

Estimated wall time:
- H100: ~6h for 17 epochs at ~6.8M states × 1.0 (with color × dihedral aug, effective 56× per state)
- G4: ~12h
- L4: ~10h

Watch the per-epoch `val_loss` (should drop monotonically through ~ep10, then plateau / slight wobble). Best checkpoint copied to Drive each epoch.

In [ ]:
%cd /content
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m alphatrain.train_path_b \
    --tensor-file alphatrain/data/v14_pillar3f.pt \
    --amp --compile \
    --resume alphatrain/data/pillar3f.pt --warm-start \
    --epochs 20 --batch-size 32768 --lr 3e-4 --warmup-epochs 1 \
    --target-temperature 0.5 \
    --copy-to /content/drive/MyDrive/alphatrain/pillar3g_best.pt \
    --save-dir /content/checkpoints/pillar3g

In [ ]:
# Copy every epoch checkpoint to Drive (for retrospective per-epoch eval)
import shutil, os, glob
DRIVE = '/content/drive/MyDrive/alphatrain'
RUN = 'pillar3g'
for f in sorted(glob.glob(f'/content/checkpoints/{RUN}/epoch_*.pt')):
    dst = f'{DRIVE}/{RUN}_{os.path.basename(f)}'
    shutil.copy(f, dst)
    print(f'Saved {dst} ({os.path.getsize(dst)/1e6:.0f} MB)')
for f in ['best.pt', 'latest.pt']:
    src = f'/content/checkpoints/{RUN}/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'{DRIVE}/{RUN}_{f}')
        print(f'Saved {DRIVE}/{RUN}_{f}')

## After pillar3g lands

Download `pillar3g_best.pt` (and intermediate epochs of interest) to M5. Then on M5:

1. Retrain value_head on pillar3g backbone (HISTORY 138): `python -m alphatrain.scripts.train_value_head --backbone alphatrain/data/pillar3g_best.pt --train-data alphatrain/data/value_targets_v14.pt --val-data alphatrain/data/value_val_K64.pt --out alphatrain/data/value_head_pillar3g.pt --epochs 5 --batch-size 8192 --lr 1e-3` (build value_targets_v14 first; EXCLUDE capped games)
2. q_weight sweep on pillar3g + value_head_pillar3g (50 seeds, MCTS@100, cap=10K, q ∈ {1.5, 2.0, 2.5})
3. Compare to pillar3f at the SAME q_weight (apples-to-apples). Decision gate: ≥+30% mean over pillar3f.
4. Floor lever (proven channel): re-mine 4800-sim corrections on pillar3g, fine-tune a crisis task-vector and merge → **pillar3h** (how pillar3f came from pillar3b).